# Anima — preliminary subject-bucket LoRA on RTX PRO 6000 Blackwell (Colab)

Finetune **CircleStone Anima** (2B anime text-to-image DiT, NVIDIA Cosmos-Predict2-2B
backbone) with **[diffusion-pipe](https://github.com/tdrussell/diffusion-pipe)**, driven by
the `geolip_anima_trainer` package. This runs a **preliminary, rank-64 `before_after` LoRA on
~1000 images** extracted into **semantic subject buckets**, and backs checkpoints up to
**HuggingFace** every few minutes (Colab is ephemeral, so the LoRA survives a disconnect).

**The methodology (why this isn't a naive run):**
- **Caption = the `task_1` JSON trained VERBATIM** (this dataset teaches JSON-prompt
  conditioning). Both `caption_vlm_json` (plain-english) and `caption_animetimm_json` (booru
  tags) are trained — *not* rendered to tags.
- **`before_after` caption mode (the first LoRA):** the two captions go into **two trees**
  (`vlm/` + `animetimm/`), trained as **two sequential phases** — the full VLM set, then the
  full animetimm set resuming the VLM adapter. (diffusion-pipe can't phase-order inside one
  run, so this is two chained runs.)
- **Subject buckets, not source buckets.** Each caption is bucketed by **its own**
  `subjects[0]`; sparse subjects are merged by **semantic similarity** (never dropped, distinct
  human subgroups kept separate); oversized buckets are **split by attribute** so none exceeds
  a data-dependent cap.
- **Anti-overtraining weighting.** `num_repeats` caps sparse buckets at ~8× (the naive
  equalize-to-largest policy would repeat a 5-image bucket ~50×/epoch → memorization).

**Runtime required:** an NVIDIA **RTX PRO 6000 Blackwell** (sm_120, 96 GB) GPU runtime — the
repo's real training target. `bf16` throughout; **no fp8, no flash-attn** (SDPA is correct on
sm_120); torch ≥ 2.7 / cu128.

> **License — read this.** Anima and *any* LoRA you train from it are **NON-COMMERCIAL**
> (CircleStone NC **+** the NVIDIA Open Model License, as a Cosmos derivative). The backup
> model card is labelled non-commercial. Keep derivatives non-commercial.

**How to run:** top to bottom. There is **one runtime restart** right after the install cell
(§2) — that is expected. Everything after the restart re-establishes its own state.

| step | section |
|---|---|
| confirm GPU | §1 |
| clone + install (→ restart) | §2 |
| verify env + `anima doctor` | §3 |
| HuggingFace auth | §4 |
| download Anima model files | §5 |
| **extract ~1000 → vlm/ + animetimm/ subject buckets** | §6 |
| **build the two phase configs (vlm, animetimm)** | §7 |
| cache latents/text-embeds (live progress) | §8 |
| `train_before_after` (two phases) + periodic HF backup | §9 |
| resume / notes | §10 |

## 1 · Confirm the GPU

You want to see **NVIDIA RTX PRO 6000 Blackwell** and ~**97 GB** of memory. If you see a T4 /
L4 / A100 instead, switch runtime (Runtime ▸ Change runtime type) — this notebook is tuned for
the Blackwell target and installs the Blackwell (cu128) torch build.

In [ ]:
!nvidia-smi

## 2 · Clone the repos and install

Clones the trainer (from GitHub, as-is) and **diffusion-pipe with submodules** (it's not
vendored on GitHub), then installs:
1. **torch + torchvision @ cu128** — required for Blackwell sm_120,
2. **diffusion-pipe's requirements** — `deepspeed`, `transformers`, `peft`, … (**no flash-attn**),
3. **light bridge deps** — `huggingface_hub`, `datasets>=2.19,<3` (the `<3` pin matters — 3.x
   changes image decoding and breaks the bridge), `Pillow`, `pyarrow`,
4. **the `[similarity]` extra** — `sentence-transformers` + `scikit-learn`, which power the
   **semantic** subject grouping in §6 (`all-MiniLM-L6-v2`, ~90 MB on first use). Without these
   the grouping silently falls back to a numpy char-trigram backend (morphological, weaker) and
   then difflib — it still never drops images, just groups them less well.

The cu128 torch and the `datasets<3` pin are re-installed **last** so neither the diffusion-pipe
requirements nor sentence-transformers can override them.

In [ ]:
import os, subprocess
WORK = "/content"
REPO = f"{WORK}/anima-trainer"
DP   = f"{REPO}/external/diffusion-pipe"
os.makedirs(WORK, exist_ok=True)

def sh(cmd):
    print("$", cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f"command failed: {cmd}")

# anima-trainer (committed GitHub state)
if not os.path.isfile(f"{REPO}/pyproject.toml"):
    sh(f"git clone --depth 1 https://github.com/AbstractEyes/anima-trainer.git {REPO}")
else:
    print("anima-trainer already present")

# diffusion-pipe is gitignored inside anima-trainer -> clone into external/ WITH submodules
if not os.path.isfile(f"{DP}/train.py"):
    sh(f"git clone --recurse-submodules https://github.com/tdrussell/diffusion-pipe.git {DP}")
else:
    print("diffusion-pipe already present")

assert os.path.isfile(f"{DP}/train.py"), "diffusion-pipe/train.py missing — clone failed"
print("OK:", f"{DP}/train.py")

In [ ]:
# 1) Blackwell torch (cu128)
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
# 2) diffusion-pipe reqs (deepspeed, transformers, peft; NO flash-attn)
!pip -q install -r {DP}/requirements.txt
# 3) bridge deps + the [similarity] extra (sentence-transformers pulls transformers/sklearn;
#    it may pin huggingface_hub<1 — harmless for the bridge). Keep on ONE line each (! magic).
!pip -q install "huggingface_hub>=0.23" "Pillow>=10" "pyarrow>=14" "sentence-transformers>=2.7,<4" "scikit-learn"
# 4) re-pin datasets<3 and cu128 torch LAST so nothing above overrode them
!pip -q install "datasets>=2.19,<3"
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
print("\n✅ installs done.")
print("⚠️  RESTART THE RUNTIME now so the new torch loads: run the next cell, or Runtime ▸ Restart session.")

**Restart the runtime.** Run the cell below (it force-restarts the kernel; Colab reconnects
automatically). After it restarts, **continue at §3** — do not re-run §2.

In [ ]:
# One-shot restart so the freshly-installed cu128 torch is the one that loads.
import os
os.kill(os.getpid(), 9)

## 3 · (after restart) Verify the environment

Re-establishes paths, puts the package on `sys.path` (we run it **from source** — no
`pip install` needed, which also sidesteps the package's Python ≥ 3.12 gate), then checks torch
sees Blackwell + cu128 + bf16, and runs `anima doctor`.

In [ ]:
import os, sys, subprocess
WORK = "/content"
REPO = f"{WORK}/anima-trainer"
DP   = f"{REPO}/external/diffusion-pipe"
sys.path.insert(0, REPO)                                   # import geolip_anima_trainer from source
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
ok_cuda = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok_cuda else "NONE"
cap  = torch.cuda.get_device_capability(0) if ok_cuda else (0, 0)
sm   = cap[0] * 10 + cap[1]
print(f"torch {torch.__version__} | cuda build {torch.version.cuda} | device {name} | sm_{sm} "
      f"| bf16 {torch.cuda.is_bf16_supported() if ok_cuda else False}")

assert ok_cuda, "No CUDA device — set a GPU runtime."
cb = tuple(int(x) for x in (torch.version.cuda or '0.0').split('.')[:2])
assert cb >= (12, 8), (f"torch cuda {torch.version.cuda} < 12.8 — re-run the §2 install cell, "
                       f"then the restart cell, then this one.")
if sm < 120:
    print(f"⚠️  Detected sm_{sm}, not Blackwell sm_120. It may run, but this notebook is tuned for "
          f"the RTX PRO 6000 Blackwell.")

In [ ]:
import geolip_anima_trainer as anima
print("geolip_anima_trainer", anima.__version__, "->", anima.__file__)

def anima_cli(*args):
    """Run the `anima` CLI via module path (no console-script / PATH / py-version dependency)."""
    cmd = [sys.executable, "-m", "geolip_anima_trainer.cli", *map(str, args)]
    print("›", " ".join(cmd))
    return subprocess.run(cmd, cwd=REPO).returncode

# Environment diagnostics: torch cu128 / sm_120 / bf16 / flash-attn absent / deepspeed / diffusion-pipe.
print(anima.doctor(repo_root=REPO).render())

## 4 · HuggingFace authentication

Add a **WRITE** token to Colab **Secrets** (🔑 in the left sidebar) named **`HF_TOKEN`**, then
run this. The token is used to read the dataset/model (public, but auth avoids rate limits) and
to **push checkpoint backups**. The backup repo is auto-named under your account.

In [ ]:
import os
def get_hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_TOKEN, "No HF_TOKEN. Add it in Colab Secrets (🔑) with WRITE scope, then re-run."

from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=True)
HF_USER = whoami(token=HF_TOKEN)["name"]
BACKUP_REPO = f"{HF_USER}/anima-prelim-1k-r64"      # <- edit if you want a different name
print("logged in as", HF_USER, "| backups ->", BACKUP_REPO)

## 5 · Download the Anima model files (~5.6 GB)

Fetches the three ComfyUI-format files into `models/anima/` via `huggingface_hub` and returns
their resolved paths (used to build the `[model]` block). Base checkpoint: **anima-base-v1.0**.

In [ ]:
MODELS_DIR = f"{REPO}/models/anima"
paths = anima.download_models(MODELS_DIR, base="base-v1.0")   # -> ModelPaths
print("transformer:", paths.transformer_path)
print("vae        :", paths.vae_path)
print("llm        :", paths.llm_path)

## 6 · Extract ~1000 images → `vlm/` + `animetimm/` subject buckets

`anima subjects` does a **columnar** pyarrow read and writes each `task_1` JSON caption
**verbatim** into its `.txt` sidecar. With `caption_mode="before_after"` (the first LoRA) it
produces **two trees** — `datasets/anima_subjects/vlm/<subject>` and `…/animetimm/<subject>` —
each caption bucketed by **its own** dominant subject. Settings resolved for `qwen_90k`:

- **caption = `caption_vlm_json` + `caption_animetimm_json`, both verbatim** (well-populated here),
- **`require_age_pass=False`** — `age_classifier_pass` is unpopulated, so the default gate drops every row,
- **audit gate ON**, **`limit=1000`** → stops after the first shard (~1.3 GB).

**Bucketing:** sparse subjects merge by **semantic similarity** into weighted `grp_*` (needs the
`[similarity]` extra from §2; else trigram/difflib fallback), large/human buckets are protected,
oversized buckets **split by attribute** (`woman·blonde_hair`) so none exceeds the data-dependent
cap, and genuine singletons pool into a weighted `misc_*` — nothing is dropped.

In [ ]:
SUBJECTS_ROOT = f"{REPO}/datasets/anima_subjects"

# Reused in §7 (build_mode_tomls reads cfg.caption_mode to emit the right toml(s)).
SUBJ_CFG = anima.SubjectBucketConfig(
    repo="AbstractPhil/diffusion-pretrain-set-ft1",
    config="qwen_90k",
    out_root=SUBJECTS_ROOT,
    limit=1000,                        # global cap -> stops after the first shard
    require_audit_approved=True,
    require_age_pass=False,            # age_classifier_pass unpopulated -> MUST be off
    caption_mode=anima.CaptionMode.BEFORE_AFTER,   # the first LoRA: two trees, two phases
    # semantic grouping (needs [similarity]; else trigram/difflib fallback)
    use_semantic=True,
    similarity_model="sentence-transformers/all-MiniLM-L6-v2",  # or nomic-...-v1 for 0 download
    min_bucket_size=10,
    min_final_group_size=8,
    keep_small=True,                   # weight, don't drop: leftovers -> misc_*
    # oversized-bucket splitting (data-dependent cap by default)
    split_oversized=True,
)
rep = anima.export_subject_buckets(SUBJ_CFG)
print(f"mode={rep['caption_mode']}  accepted={rep['accepted_images']}  dropped={rep['dropped_images']}  "
      f"raw_subjects={rep['raw_subjects']} -> buckets={rep['n_final_buckets']}  cap={rep['max_bucket_size']}")
print("caption stats:", rep["caption_stats"])
if rep["oversized_subjects"]:
    print("oversized (split):", rep["oversized_subjects"][:8])
print("\nbuckets (samples each):")
for k, v in list(rep["final_buckets"].items())[:24]:
    print(f"  {k:<30}{v}")

*(Optional)* probe the source first — caption fill rates and the audit/age gate
distributions on a small sample:

In [ ]:
# info = anima.inspect_source("AbstractPhil/diffusion-pretrain-set-ft1", "qwen_90k", n=64, verbose=True)

## 7 · Build the two phase configs (vlm, animetimm)

`build_mode_tomls` discovers the two trees and writes **`dataset_vlm.toml`** + **`dataset_animetimm.toml`**,
each with `num_repeats` balanced (diminishing-returns, `BALANCE_ALPHA=0.5`, capped at `MAX_REPEATS=8`).
Then we render **one lora toml per phase** (`lora_vlm.toml`, `lora_animetimm.toml`) pointing at its
dataset toml, with the Anima invariants enforced (**`llm_adapter_lr = 0`**, **`shuffle_caption = false`**)
and the **caching throughput knobs** set:

- **`map_num_proc`** = the image-**decode worker pool** — caching is decode-bound, and diffusion-pipe
  caps it at `min(8, cpu)`, so set this to the box's **core count** (the real cache-speed lever).
- **`caching_batch_size`** = the VAE/text-encoder batch (96 GB has headroom; 16+).

Each phase trains into its own `OUTPUT_DIR/<phase>` so phase 2 can resume phase 1's adapter.

In [ ]:
import os
from pathlib import Path

# ---- tunables (preliminary before_after LoRA) -----------------------------------
RANK                = 64           # multi-concept
LR                  = 2e-5         # Anima likes LOW lr
MICRO_BATCH         = 4            # compute-bound at 1024²; raising it grows effective batch, not samples/s
EPOCHS_VLM          = 12           # full VLM phase
EPOCHS_ANIMETIMM    = 8            # full animetimm phase (resumes the VLM adapter)
SAVE_EVERY_N_EPOCHS = 4
WARMUP_STEPS        = 50
RES                 = 1024
BALANCE_ALPHA       = 0.5          # 0=equalize(legacy/overtrains), 1=no balance, 0.5=sqrt
MAX_REPEATS         = 8
CAP_MULT            = 1.25
# throughput / VRAM (1024², single Blackwell):
#   ~2.8 samples/s at micro_batch 4 is compute-bound-NORMAL (maskless SDPA already uses the
#   efficient sm_120 backend, not math). ~89 GB VRAM is activations, not the 4 GB model — fine
#   on a 96 GB card. The ONE real speed lever is torch.compile (~+10-25%); it pays for its
#   one-time compile over a multi-epoch run, so it's on here. ACT_CKPT trades ~30% speed to free
#   VRAM for a bigger batch — leave it off unless you want micro_batch >> 4 / higher res.
COMPILE             = True         # torch.compile(dynamic=True) the DiT — the throughput win
ACT_CKPT            = False        # VRAM<->speed trade; off = faster (VRAM is abundant)
# caching throughput (decode-bound; the 2B DiT doesn't load so VRAM is low by design):
CACHING_BATCH       = 8            # VAE/text-encoder precache batch (first-shard latency, NOT speed)
MAP_NUM_PROC        = os.cpu_count() or 16   # decode workers — THE cache-speed lever (set to core count)
OUTPUT_DIR          = "/content/runs/anima_prelim"
CONFIGS_DIR         = f"{REPO}/configs/colab_prelim"
# ----------------------------------------------------------------------------------

# 1) two balanced dataset tomls (vlm tree, animetimm tree)
ds_tomls = anima.build_mode_tomls(SUBJECTS_ROOT, SUBJ_CFG, configs_dir=CONFIGS_DIR,
                                  resolutions=[RES], balance_alpha=BALANCE_ALPHA,
                                  cap_mult=CAP_MULT, max_repeats=MAX_REPEATS)
ds_tomls = {Path(t).stem: str(t) for t in ds_tomls}   # {'dataset_vlm':..., 'dataset_animetimm':...}
assert "dataset_vlm" in ds_tomls and "dataset_animetimm" in ds_tomls, ds_tomls
print("dataset tomls:", ds_tomls)

# 2) one lora toml per phase, pointing at its dataset toml
model = anima.ModelConfig(transformer_path=paths.transformer_path,
                          vae_path=paths.vae_path, llm_path=paths.llm_path,
                          llm_adapter_lr=0.0)         # FROZEN — non-negotiable for Anima

def _phase_lora(phase, ds_toml, epochs):
    run = anima.RunConfig(output_dir=f"{OUTPUT_DIR}/{phase}", epochs=epochs,
                          micro_batch_size_per_gpu=MICRO_BATCH,
                          save_every_n_epochs=SAVE_EVERY_N_EPOCHS, warmup_steps=WARMUP_STEPS,
                          compile=COMPILE, activation_checkpointing=ACT_CKPT,
                          caching_batch_size=CACHING_BATCH, map_num_proc=MAP_NUM_PROC)
    cfg = anima.TrainConfig(run=run, model=model, adapter=anima.AdapterConfig(rank=RANK),
                            optimizer=anima.OptimizerConfig(lr=LR),
                            dataset=anima.DatasetConfig(resolutions=[RES]),
                            dataset_toml_path=ds_toml)
    anima.validate(cfg)                                # enforces the Anima invariants
    p = Path(CONFIGS_DIR) / f"lora_{phase}.toml"
    p.write_text(anima.render_lora_toml(cfg), encoding="utf-8")
    return str(p)

LORA_VLM       = _phase_lora("vlm",       ds_tomls["dataset_vlm"],       EPOCHS_VLM)
LORA_ANIMETIMM = _phase_lora("animetimm", ds_tomls["dataset_animetimm"], EPOCHS_ANIMETIMM)
print("phase 1 (vlm)      :", LORA_VLM)
print("phase 2 (animetimm):", LORA_ANIMETIMM)

## 8 · Cache latents + text embeddings (both phases, live progress)

`deepspeed … train.py --cache_only` precomputes VAE latents + Qwen-3 text embeds for each
dataset (we cache **both** phase datasets). **Expect a silent warm-up of a minute or two
before the first shard appears** — diffusion-pipe first runs a *metadata pass* that opens
**every** image to read its size for aspect-ratio bucketing, and only then starts encoding
(the cache record count stays 0 until a 10 GB shard finalizes, so the live signal is **shard
bytes**, which `progress=True` reports). **Low VRAM is expected**: `--cache_only` never loads
the 2B DiT — only the VAE + Qwen-3 0.6B text encoder — so caching is **decode-bound**, which is
why §7 sets `map_num_proc` (the decode-worker count) and keeps `caching_batch_size` small.

The progress line now also **tails diffusion-pipe's own output** during the warm-up (e.g.
`Enumerating all files.`, `caching latents: …`), and the launcher sets `PYTHONUNBUFFERED=1` so
those markers stream instead of block-buffering. The first cell previews the command; then it runs.

> **A diffusion-pipe bug is auto-patched here.** diffusion-pipe's `Dataset.map` writes a metadata
> cache into an `ar_frames_*/metadata/` dir it forgets to create, which crashes caching with
> `FileNotFoundError` — and because the crash is in a forked child, the run would otherwise **hang
> forever**. The launcher injects a compat shim (`dp_compat`) that creates the dir, so this just
> works. (You need the latest repo: `!cd /content/anima-trainer && git pull` then restart the
> runtime, since the package runs from source.) If a *different* crash ever wedges caching, the
> monitor now detects the traceback and terminates instead of hanging.

> **Looks stuck?** Check it's alive with `!du -sh {OUTPUT_DIR}` in a scratch cell, or tail the
> log: `!tail -n 30 {OUTPUT_DIR}/cache_vlm.log`. A growing `metadata/` then `shard_0.bin` under
> the dataset's `cache/anima/` proves progress. Only if there's **no** movement after ~10–15 min
> suspect the known HF-`datasets`-fork hang and lower `MAP_NUM_PROC` (e.g. to 2) in §7, then re-run.

In [ ]:
anima.cache(LORA_VLM, repo_root=REPO, num_gpus=1, dry_run=True)    # preview the cache command

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)        # cache logs live here (training cell makes it too)

for phase, lora in (("vlm", LORA_VLM), ("animetimm", LORA_ANIMETIMM)):
    print(f"\n=== caching {phase} ===")
    rc = anima.cache(lora, repo_root=REPO, num_gpus=1, progress=True, progress_interval=30,
                     log_path=f"{OUTPUT_DIR}/cache_{phase}.log")   # raw deepspeed output -> the log
    print(f"cache[{phase}] exit code:", rc)

## 9 · Train both phases (`train_before_after`) + periodic HuggingFace backup

`train_before_after` runs **phase 1 (vlm)** to completion, then **phase 2 (animetimm)** resuming
phase 1's adapter via `[adapter].init_from_existing` (diffusion-pipe can't phase-order inside one
run, so this is two chained runs). We launch it in a **background thread** (so it survives the
monitor cell and a brief Colab disconnect), tee each phase to `OUTPUT_DIR/<phase>.log`, and a
second cell **backs up checkpoints to HuggingFace every few minutes**. Saved LoRAs land in
`OUTPUT_DIR/<phase>/<timestamp>/epochN/` (ComfyUI-format safetensors + PEFT config); deepspeed's
bulky `global_step*` state is skipped from the backup unless `INCLUDE_RESUME_STATE = True`.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, create_repo

INCLUDE_RESUME_STATE = False   # True = also back up deepspeed global_step* (large; needed to resume across runtimes)

_ROOT_CARD = '''---
license: other
license_name: anima-non-commercial
license_link: https://huggingface.co/circlestone-labs/Anima
base_model: circlestone-labs/Anima
tags:
- lora
- diffusion-pipe
- anima
- cosmos-predict2
- text-to-image
- non-commercial
pipeline_tag: text-to-image
---

# Anima preliminary LoRA (rank 64, before_after subject buckets)

Preliminary LoRA finetune of **CircleStone Anima** (2B DiT, NVIDIA Cosmos-Predict2-2B backbone),
trained with [diffusion-pipe](https://github.com/tdrussell/diffusion-pipe) via the
`geolip_anima_trainer` package on a ~1000-image slice of
`AbstractPhil/diffusion-pretrain-set-ft1` (`qwen_90k`).

**Methodology:** `task_1` JSON captions trained **verbatim** in **semantic subject buckets**.
`caption_mode=before_after` — a full **VLM phase** (`runs/vlm/...`) then a full **animetimm phase**
(`runs/animetimm/...`) resuming the VLM adapter. Sparse subjects grouped by semantic similarity,
oversized buckets split by attribute, weighted with a diminishing-returns `num_repeats` (capped
~8×); distinct human subgroups kept separate. `llm_adapter_lr = 0` (adapter frozen).

> **NON-COMMERCIAL.** Derivative of Anima — inherits CircleStone's non-commercial terms **and**
> the NVIDIA Open Model License (Cosmos derivative). Do not use commercially.

Each `runs/<phase>/<timestamp>/epochN/` is a ComfyUI-format LoRA — drop into `ComfyUI/models/loras/`.
'''

def _latest_run_dir(output_dir):
    """Newest <timestamp> run dir under ANY phase subtree (recursive — before_after writes
    OUTPUT_DIR/vlm/<ts>/epochN and OUTPUT_DIR/animetimm/<ts>/epochN)."""
    p = Path(output_dir)
    if not p.is_dir():
        return None
    cands = [e for e in p.rglob("epoch*") if e.is_dir() and any(e.glob("*.safetensors"))]
    cands += [g for g in p.rglob("global_step*") if g.is_dir()]
    if not cands:
        return None
    return max(cands, key=lambda d: d.stat().st_mtime).parent   # the run dir holding epoch*

def anima_hf_backup(repo_id, output_dir, *, token, include_resume_state=INCLUDE_RESUME_STATE):
    """Upload the latest run dir (epoch* LoRAs + tensorboard + config) to a HF model repo,
    preserving the phase path. Idempotent: upload_folder hashes files, unchanged ones skip."""
    run = _latest_run_dir(output_dir)
    if run is None:
        print("  [backup] no checkpoints yet under", output_dir)
        return None
    rel = run.relative_to(Path(output_dir)).as_posix()          # e.g. vlm/<timestamp>
    api = HfApi(token=token)
    create_repo(repo_id, token=token, repo_type="model", private=True, exist_ok=True)
    api.upload_file(path_or_fileobj=_ROOT_CARD.encode(), path_in_repo="README.md",
                    repo_id=repo_id, repo_type="model", commit_message="model card (non-commercial)")
    ignore = None if include_resume_state else ["global_step*/*", "global_step*/**"]
    api.upload_folder(folder_path=str(run), repo_id=repo_id, repo_type="model",
                      path_in_repo=f"runs/{rel}", ignore_patterns=ignore,
                      commit_message=f"backup :: {rel}")
    url = f"https://huggingface.co/{repo_id}/tree/main/runs/{rel}"
    print("  [backup] ->", url)
    return url

print("backup helper ready. INCLUDE_RESUME_STATE =", INCLUDE_RESUME_STATE)

**Start training** (non-blocking, both phases). `train_before_after` runs the **full VLM phase**
then the **full animetimm phase** (resuming the VLM adapter). It's launched in a daemon thread so
this cell returns immediately and the monitor cell below can back up to HuggingFace while it runs.
Each phase tees its own deepspeed output to `OUTPUT_DIR/{vlm,animetimm}.log`.

In [ ]:
import os, threading

os.makedirs(OUTPUT_DIR, exist_ok=True)

# preview both phase commands (phase 2's init_from_existing is resolved at runtime from phase 1)
anima.train_before_after(LORA_VLM, LORA_ANIMETIMM, repo_root=REPO, num_gpus=1, dry_run=True)

# run the two phases in a daemon thread; each phase logs to OUTPUT_DIR/{vlm,animetimm}.log
_train_result = {}
def _run_two_phase():
    try:
        _train_result["rc"] = anima.train_before_after(
            LORA_VLM, LORA_ANIMETIMM, repo_root=REPO, num_gpus=1,
            configs_dir=CONFIGS_DIR, log_dir=OUTPUT_DIR)
    except Exception as e:                       # surface the failure to the monitor cell
        _train_result["error"] = repr(e)
        raise

train_thread = threading.Thread(target=_run_two_phase, name="anima-train", daemon=True)
train_thread.start()
print("\ntraining started in background thread |",
      f"logs: {OUTPUT_DIR}/vlm.log -> {OUTPUT_DIR}/animetimm.log")

**Monitor + back up.** This cell tails whichever phase log is currently being written and pushes a
backup every `BACKUP_EVERY_MIN` minutes until the training thread finishes, then does a final
backup. Safe to re-run if interrupted — the training thread keeps running in the background
regardless (and `anima_hf_backup` always uploads the newest checkpoint across **both** phases).

In [ ]:
import time
from pathlib import Path

BACKUP_EVERY_MIN = 15

def _tail_active_log(output_dir, n=1500):
    """Print the tail of whichever phase log was written most recently."""
    logs = [p for p in (Path(output_dir) / "vlm.log", Path(output_dir) / "animetimm.log") if p.exists()]
    if not logs:
        return
    newest = max(logs, key=lambda p: p.stat().st_mtime)
    print(f"--- {newest.name} ---")
    print(newest.read_text(errors="replace")[-n:])

try:
    while train_thread.is_alive():
        time.sleep(BACKUP_EVERY_MIN * 60)
        _tail_active_log(OUTPUT_DIR)
        anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)
except KeyboardInterrupt:
    print("monitor stopped (training continues in background; re-run this cell to resume monitoring)")

train_thread.join(timeout=5)
if "error" in _train_result:
    print("\n=== training thread RAISED:", _train_result["error"], "===")
else:
    print("\n=== training finished | per-phase rc:", _train_result.get("rc"), "===")
_tail_active_log(OUTPUT_DIR, n=3000)
anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)   # final backup

## 10 · Resume, re-run, and notes

**Manual backup anytime** (e.g. you stopped a phase early) — uploads the newest checkpoint across
both phases:
```python
anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)
```

**Two phases, two run dirs.** `train_before_after` writes phase 1 into `OUTPUT_DIR/vlm/<ts>/` and
phase 2 into `OUTPUT_DIR/animetimm/<ts>/`. Phase 2 starts from phase 1's latest epoch (resolved at
runtime and rewritten into `[adapter].init_from_existing`), so **the animetimm adapter is the final
LoRA**. Re-running the *start training* cell restarts **both** phases from scratch.

**Resume a single phase** after an interruption (same runtime, checkpoints still on disk) — drive
that phase's lora toml directly with `resume`:
```python
anima.train(LORA_VLM,       repo_root=REPO, num_gpus=1, resume=True)   # finish the VLM phase
anima.train(LORA_ANIMETIMM, repo_root=REPO, num_gpus=1, resume=True)   # ...then the animetimm phase
```
To resume on a **fresh** runtime you need the deepspeed `global_step*` state — set
`INCLUDE_RESUME_STATE = True` in §9 so it's backed up, then download it back into the matching
`OUTPUT_DIR/<phase>/<ts>/` before resuming.

**Get the LoRA:** under `https://huggingface.co/<you>/anima-prelim-1k-r64` at
`runs/animetimm/<timestamp>/epochN/` (the final phase) — drop the `epochN` safetensors into
`ComfyUI/models/loras/`. The `runs/vlm/...` checkpoints are the intermediate phase-1 adapter.

**Caching throughput** (if §8 is slow): caching is **decode-bound**, not GPU-bound — `--cache_only`
never loads the 2B DiT, only the VAE + Qwen-3 0.6B text encoder, so **low VRAM is expected**. The
levers, set in §7: raise **`MAP_NUM_PROC`** (the image-decode worker pool — diffusion-pipe caps it at
`min(8, cpu)` unless `map_num_proc` overrides), raise **`CACHING_BATCH`** (96 GB has headroom), and on
a multi-GPU box pass `num_gpus=N` to `anima.cache`.

**Scaling up later:**
- Raise `EPOCHS_VLM` / `EPOCHS_ANIMETIMM`, and raise `SubjectBucketConfig(limit=...)` (or drop it to
  extract more shards). The full config is ~100 GB across 56 shards, so a big extraction belongs on a
  roomy disk, not Colab scratch.
- Keep `BALANCE_ALPHA=0.5` and `MAX_REPEATS=8` so the long tail of sparse buckets stays weighted down;
  lower `min_final_group_size` / loosen `sim_threshold` to fold more singletons into semantic groups
  (and shrink `misc_other`).
- Try the other caption modes from §6: `CaptionMode.SEPARATE` (both trees, one shuffled dataset.toml,
  a single `anima.train`) or `CaptionMode.MIXED` (one image on disk + multi-prompt `captions.json`).
- Keep `llm_adapter_lr = 0` unless you're deliberately A/B-testing the two-phase adapter protocol
  (ceiling 5e-6 — `validate()` hard-errors above it).

**Even more durable than backups:** point `OUTPUT_DIR` at a mounted Google Drive folder so
checkpoints persist on disk independently of the runtime:
```python
from google.colab import drive; drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/anima_runs/prelim"   # set this in §7 before building the configs
```

> **License reminder:** the LoRA is a non-commercial Anima derivative (CircleStone NC + NVIDIA
> Open Model License). The backup model card says so.